# Oriented boxes, all three classes

Oriented boxes won the comparison, so this is the whole set with oriented labels.
Measured on barriers, frame split, same data and model size either way:

| | mAP50 | mAP50-95 | inference | GFLOPs |
| --- | --- | --- | --- | --- |
| axis-aligned | 0.789 | 0.553 | 2.1 ms | 6.4 |
| oriented | **0.890** | **0.711** | **1.9 ms** | 6.6 |

+0.10 mAP50 and +0.16 mAP50-95 for no latency cost. The cause is geometric: a merged
barrier row receding from the camera is a long thin diagonal shape, and an axis-aligned
box around it is roughly 55% object and 45% empty road, which caps IoU even for a
correct prediction.

Where each class's orientation comes from:

| class | source |
| --- | --- |
| barrier | `minAreaRect` over the merged row's projected cuboid corners |
| cone | `minAreaRect` over its own cuboid corners — comes out near axis-aligned, which is the honest answer for an upright object |
| stop_sign | Task 1's labels as axis-aligned quads; COCO carries no orientation and an octagon already fills ~82% of its box |

**Two things to carry into the write-up.** This variant has no zero-shot baseline — B0
works because a COCO-pretrained model knows `stop sign`, and DOTA has no such class, so
the floor stays the axis-aligned B0 number and the comparison is cross-format. And
barrier is still on the scene split here, so expect it near zero regardless: that is the
two-scene dataset limit, not the box type. The evidence for both is in
`results/barrier_investigation.csv`; runs `B3aabb`/`B3obb`/`B4aabb`/`B4obb` in
`configs/runs.yaml` reproduce the comparison.

In [ ]:
# ---- edit this ----
NUSCENES = r"C:\path\to\nuscenes-mini"   # same path as the Task 1 notebook
EPOCHS = 100
WORKERS = 6          # drop to 2 if WinError 1455 comes back
DEVICE = ""          # "" = auto, or "cpu", or "0"
# -------------------

In [ ]:
import os, sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))

from IPython.display import Image, display

from common.config import class_names
from common.env import find_nuscenes_root, writable_root
from common.notebook import sh

NUSCENES = find_nuscenes_root(NUSCENES)
DEVICE_ARGS = ["--device", DEVICE] if DEVICE else []
OBB_ROOT = writable_root() / "datasets/yolo_3class_obb"
NAMES = class_names()
assert (REPO / "configs/data.yaml").exists(), "run the Task 1 notebook first"
print("nuScenes:", NUSCENES)

In [ ]:
# Build it. Cones and barriers from the cuboid corners; stop signs converted from
# the labels Task 1 already wrote, so the GPU pseudo-label pass is not repeated.
sh("-m", "data.obb_dataset", "--nuscenes-root", NUSCENES)

In [ ]:
# Look at the labels before training on them. Barriers should hug the row at an
# angle; cones and stop signs should look square, which is correct, not a bug.
import cv2
import numpy as np

from common.draw import color_for

dest = REPO / "results/obb_labels"
dest.mkdir(parents=True, exist_ok=True)
shown = 0
for label in sorted((OBB_ROOT / "labels" / "train").glob("*.txt")):
    lines = [l for l in label.read_text().splitlines() if l.strip()]
    if len(lines) < 2:
        continue
    img_file = next((OBB_ROOT / "images" / "train").glob(label.stem + ".*"), None)
    if img_file is None:
        continue
    img = cv2.imread(str(img_file))
    h, w = img.shape[:2]
    for line in lines:
        parts = line.split()
        cls = int(parts[0])
        pts = np.array([float(v) for v in parts[1:]]).reshape(4, 2)
        pts[:, 0] *= w
        pts[:, 1] *= h
        cv2.polylines(img, [pts.astype(np.int32)], True, color_for(cls), 2)
        cv2.putText(img, NAMES[cls], tuple(pts[0].astype(int)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_for(cls), 2, cv2.LINE_AA)
    out = dest / f"{label.stem}.jpg"
    cv2.imwrite(str(out), img)
    display(Image(str(out), width=950))
    shown += 1
    if shown == 3:
        break

In [ ]:
sh(
    "-m", "detection.train", "--run", "B5obb",
    "--set", f"epochs={EPOCHS}",
    "--set", f"workers={WORKERS}",
    "--set", "exist_ok=True",
    *DEVICE_ARGS,
)

In [ ]:
# Per class, beside the axis-aligned run. The gain is expected on barriers alone,
# so cones and stop signs staying flat is the result holding up, not a null one.
from ultralytics import YOLO

res = YOLO(str(REPO / "runs/B5obb/weights/best.pt")).val(
    data="configs/data_obb.yaml", imgsz=640, device=DEVICE or None, verbose=False
)
metric = getattr(res, "box", None) or getattr(res, "obb", None)
B1_AP50 = {"cone": 0.706, "barrier": 0.000, "stop_sign": 0.847}

print(f"B5obb   mAP50 {metric.map50:.3f}   mAP50-95 {metric.map:.3f}")
print()
print(f"{'class':<11}{'AP50':>8}{'AP50-95':>10}{'B1 AP50 (aabb)':>17}")
for i, name in enumerate(NAMES):
    try:
        print(f"{name:<11}{float(metric.ap50[i]):>8.3f}{float(metric.ap[i]):>10.3f}"
              f"{B1_AP50[name]:>17.3f}")
    except (IndexError, TypeError):
        print(f"{name:<11}{'n/a':>8}")
print()
print("OBB mAP uses polygon IoU, so this is indicative rather than strictly")
print("like-for-like; the splits do match.")

In [ ]:
# What it draws on held-out frames.
model = YOLO(str(REPO / "runs/B5obb/weights/best.pt"))
picks = [
    p for p in sorted((OBB_ROOT / "images" / "val").glob("*.jpg"))
    if (OBB_ROOT / "labels" / "val" / f"{p.stem}.txt").read_text().strip()
][:4]
for path in picks:
    res = model.predict(str(path), imgsz=640, conf=0.25, device=DEVICE or None, verbose=False)[0]
    out = REPO / f"results/obb_pred_{path.stem}.jpg"
    cv2.imwrite(str(out), res.plot())
    display(Image(str(out), width=950))

## Done when

- the drawn labels look right: barriers at an angle hugging the row, cones and stop
  signs square;
- cone and stop sign AP are close to the axis-aligned run — oriented labels should cost
  them nothing;
- barrier is near zero, as expected on the scene split, and that remains a dataset
  limitation rather than a box-type one.

`runs/B5obb/weights/best.pt` is the oriented deployable. Note for Tasks 4 and 5: an OBB
model exports through the same path, but the distance code reads `box[3]` as the ground
contact, so using this model at runtime means taking the oriented box's lowest vertex
and short side instead — which is the upgrade the oriented labels make possible.